In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
)
from sklearn.naive_bayes import GaussianNB
# from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
)

In [2]:
current_dir = Path.cwd()
utils_path = next(
    (
        p
        for p in [current_dir] + list(current_dir.parents)
        if (p / "notebook_utils.py").exists()
    ),
    None,
)

sys.path.append(str(utils_path))

In [3]:
from notebook_utils import (
    setup_env,
    load_data_for_modeling,
    get_exp_manager,
    save_sklearn_model,
    load_sklearn_model,
)

exp_manager = get_exp_manager()
PROJECT_ROOT, config = setup_env()
df = load_data_for_modeling(config, PROJECT_ROOT, data_source="szfo_df")
cat_cols = config['features']['cat_cols']
target_col = config['features']['target_col']
target_mode = config['experiment']['target_mode']

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=[target_col]), 
    df[target_col], 
    test_size=0.2, 
    random_state=42, 
    stratify=df[target_col]
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Загрузка данных из: D:\Education\Arcticle\dtp_project\data\processed\dtp_szfo.parquet
Режим таргета: binary_severe. Распределение:
target
0    0.6065
1    0.3935
Name: proportion, dtype: float64
Train: (133959, 87), Test: (33490, 87)


In [4]:
CURRENT_STAGE = "02_simple_models" 

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), 
            ('scaler', StandardScaler())
        ]), [c for c in X_train.columns if c not in cat_cols]),
        
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ]
)

models_config = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1, random_state=42),
    'SGD_Log': SGDClassifier(loss='log_loss', penalty='l2', class_weight='balanced', n_jobs=-1, random_state=42), # Аналог LogReg
    'SGD_SVM': SGDClassifier(loss='hinge', penalty='l2', class_weight='balanced', n_jobs=-1, random_state=42),    # Аналог LinearSVM (быстрый)
    
    'GaussianNB': GaussianNB(),
    
    # 'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1), # Раскомментируй, если есть время/RAM
    
    'DecisionTree': DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=42),
    
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=42),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=100, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=42),
    
    'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42, algorithm='SAMME'),
}
print(f"Подготовлено к обучению: {len(models_config)} моделей.")

Подготовлено к обучению: 8 моделей.


In [6]:
results = []

for name, model in models_config.items():
    print(f">>> Training {name}...")
    
    clf = Pipeline([
        ('preprocessor', preprocessor),
        ('clf', model)
    ])
    
    try:
        clf.fit(X_train, y_train)

        save_sklearn_model(clf, exp_manager, model_name=name, stage=CURRENT_STAGE)
        
        y_pred = clf.predict(X_test)
        
        try:
            if hasattr(clf, "predict_proba"):
                y_proba = clf.predict_proba(X_test)
            elif hasattr(clf['clf'], "predict_proba"):
                y_proba = clf['clf'].predict_proba(clf['preprocessor'].transform(X_test))
            else:
                y_proba = None
                
            if y_proba is not None:
                if y_test.nunique() == 2:
                    auc = roc_auc_score(y_test, y_proba[:, 1])
                else:
                    auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            else:
                auc = 0.5
        except Exception as e:
            auc = 0.5

        f1_macro = f1_score(y_test, y_pred, average='macro')
        recall_macro = recall_score(y_test, y_pred, average='macro')
        precision_macro = precision_score(y_test, y_pred, average='macro')
        acc = accuracy_score(y_test, y_pred)
        
        if y_test.nunique() == 2:
            f1_pos = f1_score(y_test, y_pred, pos_label=1)
            recall_pos = recall_score(y_test, y_pred, pos_label=1)
            precision_pos = precision_score(y_test, y_pred, pos_label=1)
        else:
            f1_pos = recall_pos = precision_pos = None

        print(f"Done. F1 Macro: {f1_macro:.4f} | Recall Macro: {recall_macro:.4f} | AUC: {auc:.4f}")
        
        results.append({
            'Model': name, 
            'Accuracy': acc,
            'F1_Macro': f1_macro,
            'Recall_Macro': recall_macro,
            'Precision_Macro': precision_macro,
            'ROC_AUC': auc,
            'F1_Positive': f1_pos,
            'Recall_Positive': recall_pos,
            'Precision_Positive': precision_pos
        })
        
    except Exception as e:
        print(f"!!! Error training {name}: {e}")

>>> Training LogisticRegression...
Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\LogisticRegression\model.joblib
Done. F1 Macro: 0.6568 | Recall Macro: 0.6627 | AUC: 0.7249
>>> Training SGD_Log...
Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\SGD_Log\model.joblib
Done. F1 Macro: 0.6518 | Recall Macro: 0.6571 | AUC: 0.7145
>>> Training SGD_SVM...
Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\SGD_SVM\model.joblib
Done. F1 Macro: 0.6505 | Recall Macro: 0.6555 | AUC: 0.5000
>>> Training GaussianNB...
Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\GaussianNB\model.joblib
Done. F1 Macro: 0.5887 | Recall Macro: 0.5922 | AUC: 0.6591
>>> Training DecisionTree...
Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\DecisionTree\model.joblib
Done. F1 Macro: 0.6159 | Recall Macro

d:\Education\Arcticle\dtp_project\.venv\lib\site-packages\sklearn\ensemble\_weight_boosting.py:519: FutureWarning: The parameter 'algorithm' is deprecated in 1.6 and has no effect. It will be removed in version 1.8.
  warnings.warn(


Модель сохранена: D:\Education\Arcticle\dtp_project\res\01_dtp_stat\models\02_simple_models\AdaBoost\model.joblib
Done. F1 Macro: 0.5734 | Recall Macro: 0.5828 | AUC: 0.6758


In [7]:
results_df = pd.DataFrame(results).sort_values(by='F1_Macro', ascending=False)
stage_paths = exp_manager.get_paths(stage=CURRENT_STAGE)
results_csv_path = stage_paths['report'] / "metrics_comparison.csv"
results_df.to_csv(results_csv_path, index=False)
results_df.style.background_gradient(cmap='viridis')

,Model,Accuracy,F1_Macro,Recall_Macro,Precision_Macro,ROC_AUC,F1_Positive,Recall_Positive,Precision_Positive
0,LogisticRegression,0.664467,0.656808,0.662708,0.656482,0.724937,0.605539,0.654450,0.563431
1,SGD_Log,0.659928,0.651773,0.657141,0.651327,0.714458,0.598484,0.644055,0.558936
2,SGD_SVM,0.658973,0.650491,0.655528,0.649932,0.500000,0.596046,0.639350,0.558235
5,RandomForest,0.645536,0.640846,0.650711,0.643911,0.707925,0.599804,0.675013,0.539675
6,ExtraTrees,0.643207,0.637001,0.644595,0.638363,0.703496,0.589537,0.651112,0.538602
4,DecisionTree,0.618125,0.615907,0.630537,0.625082,0.676923,0.586718,0.688823,0.510976
3,GaussianNB,0.641893,0.588728,0.592182,0.618307,0.659095,0.440860,0.358753,0.571705
7,AdaBoost,0.641415,0.573441,0.582836,0.620314,0.675807,0.403161,0.307762,0.584270


In [ ]:
def get_top_features(pipeline, X_train, cat_cols, top_n=15):
    """Возвращает список топ-N признаков из обученного пайплайна."""
    try:
        preprocessor = pipeline.named_steps['preprocessor']
        model = pipeline.named_steps['clf']
        
        num_cols = [c for c in X_train.columns if c not in cat_cols]
        
        # Правильный доступ к трансформерам
        cat_transformer = preprocessor.named_transformers_['cat']
        ohe = cat_transformer.named_steps['onehot']
        ohe_features = ohe.get_feature_names_out(cat_cols)
        
        feature_names = list(num_cols) + list(ohe_features)
        
        # Получаем важности
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
        elif hasattr(model, 'coef_'):
            importances = np.abs(model.coef_[0]) if model.coef_.ndim > 1 else np.abs(model.coef_)
        else:
            return []
        
        if len(importances) != len(feature_names):
            return []
        
        # Сортируем и берём топ-N
        indices = np.argsort(importances)[::-1][:top_n]
        return [feature_names[i] for i in indices]
    
    except Exception as e:
        print(f"⚠️ Ошибка при извлечении топ признаков: {e}")
        return []

# === 1. Собираем топ-15 признаков для каждой модели ===
models_to_inspect = ['RandomForest', 'ExtraTrees', 'LogisticRegression', 'DecisionTree', 'AdaBoost', 'SGD_Log']
top_features_per_model = {}

for name in models_to_inspect:
    full_name = f"{name}_{target_mode}.joblib"
    try:
        print(f"Загрузка {name}...", end=' ')
        model = load_sklearn_model(full_name, config, PROJECT_ROOT)
        top_feats = get_top_features(model, X_train, cat_cols, top_n=15)
        top_features_per_model[name] = set(top_feats)
        print(f"✓ ({len(top_feats)} признаков)")
    except Exception as e:
        print(f"✗ Ошибка: {e}")
        top_features_per_model[name] = set()

# === 2. Строим консенсусную таблицу ===
# Собираем все уникальные признаки из топ-15 всех моделей
all_features = set()
for feats in top_features_per_model.values():
    all_features.update(feats)

consensus_df = pd.DataFrame(index=sorted(all_features), columns=models_to_inspect)

for model_name, feature_set in top_features_per_model.items():
    for feature in all_features:
        consensus_df.loc[feature, model_name] = '+' if feature in feature_set else ''

# Считаем количество моделей, где признак в топ-15 (более надёжный способ)
consensus_df['total_models'] = (consensus_df == '+').sum(axis=1)

# === 3. Преобразуем индекс в колонку и сортируем ===
consensus_df = consensus_df.reset_index().rename(columns={'index': 'feature'})
consensus_df = consensus_df.sort_values(['total_models', 'feature'], ascending=[False, True]).reset_index(drop=True)

# === 4. Выводим красивую таблицу ===
print("\n" + "="*80)
print("КОНСЕНСУСНАЯ ТАБЛИЦА: ПРИЗНАКИ В ТОП-15 ПО МОДЕЛЯМ")
print("="*80)
print(f"Всего уникальных признаков в топ-15: {len(consensus_df)}")
print(f"Модели: {', '.join(models_to_inspect)}\n")

# Оставляем только признаки, которые есть хотя бы в 2 моделях (убираем шум)
display_df = consensus_df[consensus_df['total_models'] >= 2].copy()

# Переименовываем колонки моделей для компактности
col_rename = {
    'RandomForest': 'RandF',
    'ExtraTrees': 'ExtrT',
    'LogisticRegression': 'LogR',
    'DecisionTree': 'DecT',
    'AdaBoost': 'AdaB',
    'SGD_Log': 'SGD'
}
display_df = display_df.rename(columns=col_rename)

# Форматируем вывод: оставляем только нужные колонки
cols_to_show = list(col_rename.values()) + ['total_models']
final_df = display_df[['feature'] + cols_to_show].copy()

# Выводим таблицу
final_df.to_string(index=False)

Загрузка RandomForest... ✓ (15 признаков)
Загрузка ExtraTrees... ✓ (15 признаков)
Загрузка LogisticRegression... ✓ (15 признаков)
Загрузка DecisionTree... ✓ (15 признаков)
Загрузка AdaBoost... ✓ (15 признаков)
Загрузка SGD_Log... ✓ (15 признаков)

КОНСЕНСУСНАЯ ТАБЛИЦА: ПРИЗНАКИ В ТОП-15 ПО МОДЕЛЯМ
Всего уникальных признаков в топ-15: 43
Модели: RandomForest, ExtraTrees, LogisticRegression, DecisionTree, AdaBoost, SGD_Log

                                         feature RandF ExtrT LogR DecT AdaB SGD  total_models
       region_id_санктпетербург_пушкинский_район     +     +    +    +    +   +             6
region_id_санктпетербург_красногвардейский_район           +    +    +    +   +             5
                            light_cat_night_dark     +     +         +    +                 4
                            nearby_objects_count     +     +         +    +                 4
      region_id_санктпетербург_центральный_район           +    +         +   +             4
          